In [1]:
from statsmodels.stats.multitest import multipletests
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from pyliftover import LiftOver
from collections import Counter
from pyfaidx import Fasta
import matplotlib as mpl
import seaborn as sns
import pandas as pd
import polars as pl
import statsmodels
import numpy as np
import os

In [2]:
plt.rcParams["pdf.fonttype"] = 42  # Use TrueType fonts instead of Type 3 fonts
plt.rcParams["ps.fonttype"] = 42  # For PostScript as well, if needed

# Linkage Regions From Scratch

### Download all needed data

**GWAS Summary Statistics**

In [3]:
GWAS = pd.read_csv("../2_source/GWAS_raw.tsv", sep='\t')
GWAS['logp'] = -np.log(GWAS['p_value'])         # Extract log p value
GWAS['sig'] = GWAS['p_value'] < 5*10**-8               # Mark significant loci
GWAS.to_csv("../2_source/GWAS.tsv", sep='\t', index=None)

**GWAS Leads**

In [4]:
leads = pd.read_excel('../2_source/GP2_leads.xlsx', sheet_name='Table S3')
leads.to_csv('../2_source/GP2_leads.tsv', sep='\t', index=None)

**Donor variant bcf**

### PLINK

**Download plink2**

**Download and decompress hg38.pgen:**

**Download and decompress hg38.pvar:**

**Download hg38.psam:**

**Create list of lead rs values**

**Use plink to calculate r² values between GP2 lead variants and 1000G variants**

**Load GWAS**

In [5]:
GWAS = pd.read_csv("../2_source/GWAS.tsv", sep='\t')
print(f'Rows: {GWAS.shape[0]}')
GWAS.sample(3)

Rows: 11714528


,chromosome,base_pair_position,SNP_ID,effect_allele,other_allele,effect_allele_frequency,N_datasets,p_value,beta,standard_error,p_value(random),beta(random),I,logp,sig
2008436,3,34957595,chr3:34957595:T:C,C,T,0.0148,19,0.4110,-0.0246,0.029922,0.4080,-0.0249,0.51,0.889162,False
4199738,5,174297128,chr5:174297128:T:C,C,T,0.1107,21,0.9734,-0.0004,0.011996,0.9734,-0.0004,0.00,0.026960,False
8187310,12,70547530,chr12:70547530:T:C,C,T,0.0440,3,0.5744,-0.0166,0.029559,0.5744,-0.0166,0.00,0.554429,False


### Define regions

**Load Leads**

In [6]:
leads = pd.read_csv('../2_source/GP2_leads.tsv', sep='\t')
print(f'Rows: {leads.shape[0]}')
leads.sample(3)

Rows: 157


,SNP ID,rsID,Known or Novel,CHR,BP,Nearest Gene,Reference allele,Alternative allele,Effect allele,Effect allele frequency,...,"Beta, case-control","SE, case-control","P, case-control","Freq, biobank","Beta, biobank","SE, biobank","P, biobank","I2, all studies","Passes, COJO",Locus Number
152,chr21:37519947:A:G,rs11701836,Known,21,37519947,DYRK1A,A,G,G,0.2636,...,0.0901,0.013316,1.320000e-11,0.2742,0.0516,0.00915,1.700000e-08,16.41,YES,130
29,chr3:17710565:A:G,rs144678625,Known,3,17710565,TBC1D5,A,G,G,0.0139,...,0.1361,0.053090,1.040000e-02,0.0137,0.1818,0.03670,7.380000e-07,0.00,YES,25
125,chr17:4932600:T:C,rs2243093,Novel,17,4932600,GP1BA,T,C,C,0.1304,...,0.0656,0.017633,1.990000e-04,0.1397,0.0554,0.01160,1.920000e-06,44.45,YES,106


**Load plink output**

In [7]:
r2 = pd.read_csv('../1_plink/plink_clean_005.vcor', sep='\t')
print('A: GP2 variants, B: 1kG variant')
print(f'Rows: {r2.shape[0]}')
r2.sample(3)

A: GP2 variants, B: 1kG variant
Rows: 93528


,#CHROM_A,POS_A,ID_A,CHROM_B,POS_B,ID_B,PHASED_R2
91356,20,31805133,rs35428411,20,33220056,rs61340354,0.165838
53275,9,34154849,rs13296299,9,34347341,rs36034872,0.206946
34248,6,27287064,rs6913724,6,26886000,rs12182179,0.051665


### Demarcate lead-variant centered regions by different r² threshold

In [8]:
regions = []
for lead_rs in set(leads['rsID']):
    for threshold in [4,5,6,7,8]:

        # If lead rs in subset of 1000G
        if lead_rs in set(r2['ID_A']):
    
            # If lead rs in 1kg AND has a at least one link at this rs threshold -------------------------------
            if lead_rs in set(r2[r2['PHASED_R2'] > threshold/10]['ID_A']):
                r2_block = r2[(r2['ID_A'] == lead_rs) & (r2['PHASED_R2'] > threshold/10)]
                chr = r2_block['#CHROM_A'].values[0]
                lead = r2_block['POS_A'].values[0]
                left_border = min([lead,min(r2_block['POS_B'])])
                right_border = max([lead,max(r2_block['POS_B'])])
                LD_left = max([0,r2_block['POS_A'].values[0] - left_border])
                LD_right = max([0,right_border - r2_block['POS_A'].values[0]])
                regions.append({
                    'chr':chr,
                    'rs_lead':lead_rs,
                    'lead_hg38':lead,
                    'left_raw':left_border,
                    'right_raw':right_border,
                    'left_dist':LD_left,
                    'right_dist':LD_right,
                    'size':LD_left + LD_right,
                    'r2_threshold_x10':threshold,
                    '1kg':True
                })
                
            # If lead rs in 1kg AND has NO link at this rs threshold -------------------------------------------
            else:
                r2_block = r2[(r2['ID_A'] == lead_rs)]
                chr = r2_block['#CHROM_A'].values[0]
                lead = r2_block['POS_A'].values[0]
                regions.append({
                    'chr':chr,
                    'rs_lead':lead_rs,
                    'lead_hg38':lead,
                    'left_raw':lead,
                    'right_raw':lead,
                    'left_dist':0,
                    'right_dist':0,
                    'size':0,
                    'r2_threshold_x10':threshold,
                    '1kg':True
                })

        # If lead rs NOT in 1kg --------------------------------------------------------------------------------
        else:
            chr = leads[leads['rsID'] == lead_rs]['CHR'].values[0]
            lead = leads[leads['rsID'] == lead_rs]['BP'].values[0]
            
            regions.append({
                'chr':chr,
                'rs_lead':lead_rs,
                'lead_hg38':lead,
                'left_raw':lead,
                'right_raw':lead,
                'left_dist':0,
                'right_dist':0,
                'size':0,
                'r2_threshold_x10':threshold,
                '1kg':False
            })

In [9]:
regions_raw = pd.DataFrame(regions)
print(f'Rows: {regions_raw.shape[0]}')
regions_raw.sample(3)

Rows: 785


,chr,rs_lead,lead_hg38,left_raw,right_raw,left_dist,right_dist,size,r2_threshold_x10,1kg
175,17,rs2243093,4932600,4918697,4995935,13903,63335,77238,4,True
318,1,rs708723,205770138,205750404,205770138,19734,0,19734,7,True
205,7,rs2127183,95711872,95652761,95715240,59111,3368,62479,4,True


In [10]:
regions_raw.to_csv('../4_regions/regions_0_raw.tsv', sep='\t', index=False)

### Choose r² threshold 0.6

In [11]:
regions_hg38 = pd.read_csv('../4_regions/regions_0_raw.tsv', sep='\t')
print(f'Rows: {regions_hg38.shape[0]}')
regions_hg38.sample(3)

Rows: 785


,chr,rs_lead,lead_hg38,left_raw,right_raw,left_dist,right_dist,size,r2_threshold_x10,1kg
245,5,rs11241774,124872055,124868255,124953598,3800,81543,85343,4,True
685,1,rs377808,52724687,52663447,52973853,61240,249166,310406,4,True
140,4,rs13117238,76252761,76225920,76261256,26841,8495,35336,4,True


**Create shortened id for later joining**

In [12]:
regions_hg38['hg38id_short'] = ['chr']*len(regions_hg38) + regions_hg38['chr'].astype(str) + \
    ['_']*len(regions_hg38) + regions_hg38['lead_hg38'].astype(str)

**Subset to threshold 0.6**

In [13]:
regions_hg38_r6 = regions_hg38[(regions_hg38['r2_threshold_x10'] == 6) | (regions_hg38['r2_threshold_x10'] == 0)].copy()
print(f'Rows: {regions_hg38_r6.shape[0]}')
pd.concat([regions_hg38_r6[regions_hg38_r6['1kg'] == True].sample(2),regions_hg38_r6[regions_hg38_r6['1kg'] == False].sample(2)])

Rows: 157


,chr,rs_lead,lead_hg38,left_raw,right_raw,left_dist,right_dist,size,r2_threshold_x10,1kg,hg38id_short
212,12,rs61754230,71785666,71785666,71785666,0,0,0,6,True,chr12_71785666
412,2,rs9309428,69465874,69300398,69516134,165476,50260,215736,6,True,chr2_69465874
622,18,rs2076574176,34050285,34050285,34050285,0,0,0,6,False,chr18_34050285
742,1,rs76763715,155235843,155235843,155235843,0,0,0,6,False,chr1_155235843


### Apply lower bound of 1.5kbp at each side of the lead variant

In [14]:
buffer = 1500

In [15]:
regions_hg38_r6['left_bound'] = regions_hg38_r6.apply(lambda r: r['lead_hg38'] - buffer if r['lead_hg38'] - r['left_raw'] < 1500 else r['left_raw'], axis=1)
regions_hg38_r6['right_bound'] = regions_hg38_r6.apply(lambda r: r['lead_hg38'] + buffer if r['right_raw'] - r['lead_hg38'] < 1500 else r['right_raw'], axis=1)
regions_hg38_r6['left_dist'] = regions_hg38_r6.apply(lambda r: r['lead_hg38'] - r['left_bound'], axis=1)
regions_hg38_r6['right_dist'] = regions_hg38_r6.apply(lambda r: r['right_bound'] - r['lead_hg38'], axis=1)
regions_hg38_r6['size'] = regions_hg38_r6.apply(lambda r: r['right_bound'] - r['left_bound'], axis=1)

In [16]:
print(f'Rows: {regions_hg38_r6.shape[0]}')
pd.concat([regions_hg38_r6[regions_hg38_r6['1kg'] == True].sample(2),regions_hg38_r6[regions_hg38_r6['1kg'] == False].sample(2)])

Rows: 157


,chr,rs_lead,lead_hg38,left_raw,right_raw,left_dist,right_dist,size,r2_threshold_x10,1kg,hg38id_short,left_bound,right_bound
337,4,rs75739600,169996341,169965443,170070787,30898,74446,105344,6,True,chr4_169996341,169965443,170070787
172,17,rs847680,50146710,50123470,50159289,23240,12579,35819,6,True,chr17_50146710,50123470,50159289
507,3,rs1719750116,151946018,151946018,151946018,1500,1500,3000,6,False,chr3_151946018,151944518,151947518
742,1,rs76763715,155235843,155235843,155235843,1500,1500,3000,6,False,chr1_155235843,155234343,155237343


### Add nearest gene

In [17]:
regions_hg38_with_gene = regions_hg38_r6.merge(leads[['rsID','Nearest Gene']], left_on='rs_lead', right_on='rsID'
                                              ).rename(columns={'Nearest Gene':'nearest_gene'}).drop(columns=['r2_threshold_x10','rsID'])

In [18]:
print(f'Rows: {regions_hg38_with_gene.shape[0]}')
regions_hg38_with_gene.sample(3)

Rows: 157


,chr,rs_lead,lead_hg38,left_raw,right_raw,left_dist,right_dist,size,1kg,hg38id_short,left_bound,right_bound,nearest_gene
117,3,rs56384862,58410136,58276014,58484624,134122,74488,208610,True,chr3_58410136,58276014,58484624,PXK
55,1,rs10495249,226731418,226725055,226737174,6363,5756,12119,True,chr1_226731418,226725055,226737174,ITPKB
32,6,rs41316625,29015989,28895821,29195590,120168,179601,299769,True,chr6_29015989,28895821,29195590,ZNF311


In [19]:
regions_hg38_with_gene.to_csv('../4_regions/regions_1_r6_bounded.tsv', sep='\t', index=False)

### Lift regions from hgg38 to hs1/T2T/chm13

In [20]:
regions = pd.read_csv('../4_regions/regions_1_r6_bounded.tsv', sep='\t')
print(f'Rows: {regions.shape[0]}')
regions.sample(3)

Rows: 157


,chr,rs_lead,lead_hg38,left_raw,right_raw,left_dist,right_dist,size,1kg,hg38id_short,left_bound,right_bound,nearest_gene
53,13,rs1198499,49341314,49341314,49390921,1500,49607,51107,True,chr13_49341314,49339814,49390921,CAB39L
63,1,rs708723,205770138,205750404,205788696,19734,18558,38292,True,chr1_205770138,205750404,205788696,RAB29
103,2,rs76116224,17966582,17966582,18016118,1500,49536,51036,True,chr2_17966582,17965082,18016118,KCNS3


**Load Liftover Object**

In [21]:
lo = LiftOver('hg38','Hs1')

In [22]:
fail = 0
def liftSafe(chr, bp, chain):
    result = chain.convert_coordinate(chr, bp)

    if result:
        return int(result[0][1])
    else:
        print(chr, bp, 'failed')
        global fail
        fail = fail + 1
        return -1

**Perform lift**

In [23]:
fail = 0
regions_t2t = regions.copy()
regions_t2t['lead_hs1'] = regions_t2t.apply(lambda r: liftSafe(f"chr{r['chr']}", r['lead_hg38'], lo), axis=1)
regions_t2t['chr'] = ['chr']*len(regions_t2t) + regions_t2t['chr'].astype(str)
regions_t2t['lift_success'] = regions_t2t.apply(lambda r: True if r['lead_hs1'] > 0 else False, axis=1)
regions_t2t['lead_hs1'] = regions_t2t.apply(lambda r: r['lead_hg38'] if r['lead_hs1'] < 0 else r['lead_hs1'], axis=1)

chr14 37749513 failed
chr10 119856143 failed


In [24]:
print(f'Rows: {regions_t2t.shape[0]}')
pd.concat([regions_t2t[regions_t2t['lift_success'] == True].sample(3), regions_t2t[regions_t2t['lift_success'] == False]])

Rows: 157


,chr,rs_lead,lead_hg38,left_raw,right_raw,left_dist,right_dist,size,1kg,hg38id_short,left_bound,right_bound,nearest_gene,lead_hs1,lift_success
74,chr15,rs34631560,88973692,88922303,88978760,51389,5068,56457,True,chr15_88973692,88922303,88978760,MFGE8,86728586,True
66,chr17,rs72843781,62016181,61886550,62065845,129631,49664,179295,True,chr17_62016181,61886550,62065845,MED13,62885128,True
81,chr20,rs143230159,6005604,6003768,6032395,1836,26791,28627,True,chr20_6005604,6003768,6032395,CRLS1,6045914,True
9,chr14,rs36003317,37749513,37729985,37760887,19528,11374,30902,True,chr14_37749513,37729985,37760887,TTC6,37749513,False
14,chr10,rs1174256174,119856143,119856143,119856143,1500,1500,3000,False,chr10_119856143,119854643,119857643,CASC2,119856143,False


**T2T regions based on lift and sizes**

In [25]:
regions_t2t['left_hs1'] = regions_t2t['lead_hs1'] - regions_t2t['left_dist']
regions_t2t['right_hs1'] = regions_t2t['lead_hs1'] + regions_t2t['right_dist']

In [26]:
regions_t2t[['chr','left_hs1','right_hs1','lead_hs1','rs_lead','lift_success','1kg','nearest_gene']].to_csv(
    '../4_regions/regions_2_lifted.tsv', sep='\t', index=None)

### Subset donor variants by regions

### group variant to region per variant

In [27]:
variants_mapped = pd.read_csv('../3_variants/donor_subset.tsv', sep='\t')
print(f'Rows: {variants_mapped.shape[0]}')
variants_mapped.sample(3)

Rows: 168647


,chr,T2T,ref,alt,start,stop,lead,rs_lead,lift_success,in_1kg,nearest_gene
93766,chr12,914652,C,A,867499,938441,902043,rs2286028,True,True,WNK1
6770,chr1,154566840,G,A,154079804,155062748,154374933,rs2230288,True,True,GBA1
130397,chr17,43464604,C,G,43406185,43523696,43485959,rs12601457,True,True,TUBG1


In [28]:
variants_mapped['variant_id_hs1'] = variants_mapped.apply(lambda r: r['chr'] + str(r['T2T']) + r['ref'] + r['alt'], axis=1)

In [29]:
grouped = variants_mapped.groupby(by='variant_id_hs1')
donor_variants = pd.DataFrame({
    'rs_lead':grouped['rs_lead'].apply(list),
}).reset_index(drop=False)

In [30]:
counts = Counter([len(lst[1]) for lst in donor_variants.values])
print(f'There are {counts[1]} variants in one region only')
print(f'There are {counts[2]} variants in two regions at once')
print(f'There are {counts[3]} variants in three regions at once')
print(f'There are {counts[4]} variants in four regions at once')
print(f'There are {counts[5]} variants in five regions at once')

There are 158910 variants in one region only
There are 4618 variants in two regions at once
There are 167 variants in three regions at once
There are 0 variants in four regions at once
There are 0 variants in five regions at once


In [31]:
donor_variants['region_1'] = [lst[1][0] for lst in donor_variants.values]
donor_variants['region_2'] = [lst[1][1] if len(lst[1]) > 1 else 'none' for lst in donor_variants.values]
donor_variants['region_3'] = [lst[1][2] if len(lst[1]) > 2 else 'none' for lst in donor_variants.values]

In [32]:
donor_variants = donor_variants.drop(columns=['rs_lead'])

In [33]:
print(f'Rows: {donor_variants.shape[0]}')
pd.concat([
    donor_variants[donor_variants['region_2'] == 'none'].sample(1),
    donor_variants[(donor_variants['region_2'] != 'none') & (donor_variants['region_3'] == 'none')].sample(1),
    donor_variants[donor_variants['region_3'] != 'none'].sample(1)])

Rows: 163695


,variant_id_hs1,region_1,region_2,region_3
161586,chr934190464GA,rs13296299,none,none
24184,chr1240299073AG,rs35303786,rs17443099,none
8105,chr1154375275GC,rs75548401,rs76763715,rs2230288


In [34]:
donor_variants[['variant_id_hs1','region_1','region_2','region_3']].to_csv(
    '../3_variants/donor_variants_regions.tsv', sep='\t', index=None)